In [ ]:
# Downloaded most recent advanced stats

import pandas as pd
from pathlib import Path
from datetime import datetime, date

import numpy as np
from pathlib import Path

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

MONEYPuck_RAW_FILE = DATA_DIR / "moneypuck_all_teams_raw.csv"

MONEYPuck_URL = "https://moneypuck.com/moneypuck/playerData/careers/gameByGame/all_teams.csv"


def download_moneypuck(force: bool = False) -> Path:
    """
    Download the MoneyPuck all_teams.csv file if it doesn't exist locally
    or if force=True. Returns the local filepath.
    """
    if MONEYPuck_RAW_FILE.exists() and not force:
        print(f"Using cached MoneyPuck file: {MONEYPuck_RAW_FILE}")
        return MONEYPuck_RAW_FILE

    print("Downloading MoneyPuck all_teams.csv...")
    df = pd.read_csv(MONEYPuck_URL)
    df.to_csv(MONEYPuck_RAW_FILE, index=False)
    print(f"Saved MoneyPuck data to {MONEYPuck_RAW_FILE}")
    return MONEYPuck_RAW_FILE

download_moneypuck(force=True)
df = pd.read_csv("data/moneypuck_all_teams_raw.csv")

In [4]:
import psutil, os

def show_mem():
    used_gb = psutil.Process(os.getpid()).memory_info().rss / 1024**3
    print(f"Current memory usage: {used_gb:.2f} GB")


In [8]:
import pandas as pd
import numpy as np
from pathlib import Path

# =========================
# CONFIG
# =========================
DATA_DIR       = Path("data")
MONEYPUCK_FILE = DATA_DIR / "moneypuck_all_teams_raw.csv"   # your raw MoneyPuck export
OUTPUT_FILE    = DATA_DIR / "master.csv"                    # stats-only, game-level master

# Rolling horizons (games)
ROLL_WINDOWS = [3, 7, 15, 30]

# EWMs (fast/slow)
EWM_ALPHAS = {
    "fast": 0.3,   # emphasizes last ~3–4 games
    "slow": 0.1,   # emphasizes last ~10+ games
}

# =========================
# STEP 1: LOAD MONEYPUCK RAW & FILTER TEAM-LEVEL
# =========================
mp = pd.read_csv(MONEYPUCK_FILE)
mp["gameDate"] = pd.to_datetime(mp["gameDate"].astype(str), format="%Y%m%d", errors="coerce")

# Only team-level, aggregated over all situations
mp_team = mp[
    (mp["position"] == "Team Level") &
    (mp["situation"].str.lower() == "all")
].copy()

print("Team-level rows after filter:", mp_team.shape)
print(mp_team[["season", "gameId"]].drop_duplicates().groupby("season")["gameId"].nunique().head())



show_mem()
print('step 1 done')
# =========================
# STEP 2–3: KEEP ID + OPTIMIZED STAT COLUMNS
# =========================
id_cols = [
    "gameId", "gameDate", "season", "team", "playerTeam", "opposingTeam",
    "home_or_away", "playoffGame"
]

# Optimized list of useful stats for ML / spreads / totals
KEEP_STAT_COLS = [
    # Scoreboard / shots
    "goalsFor", "goalsAgainst",
    "shotsOnGoalFor", "shotsOnGoalAgainst",
    "shotAttemptsFor", "shotAttemptsAgainst",
    "blockedShotAttemptsFor", "blockedShotAttemptsAgainst",
    "missedShotsFor", "missedShotsAgainst",
    "highDangerShotsFor", "highDangerShotsAgainst",
    "mediumDangerShotsFor", "mediumDangerShotsAgainst",
    "lowDangerShotsFor", "lowDangerShotsAgainst",
    "totalShotCreditFor", "totalShotCreditAgainst",
    "scoreAdjustedTotalShotCreditFor", "scoreAdjustedTotalShotCreditAgainst",

    # xG and adjustments
    "xGoalsFor", "xGoalsAgainst",
    "flurryAdjustedxGoalsFor", "flurryAdjustedxGoalsAgainst",
    "scoreVenueAdjustedxGoalsFor", "scoreVenueAdjustedxGoalsAgainst",

    # Percentages
    "xGoalsPercentage", "corsiPercentage", "fenwickPercentage",

    # Danger xG
    "lowDangerxGoalsFor", "lowDangerxGoalsAgainst",
    "mediumDangerxGoalsFor", "mediumDangerxGoalsAgainst",
    "highDangerxGoalsFor", "highDangerxGoalsAgainst",

    # Discipline / physicality / puck control
    "penaltiesFor", "penaltiesAgainst",
    "penalityMinutesFor", "penalityMinutesAgainst",
    "hitsFor", "hitsAgainst",
    "takeawaysFor", "takeawaysAgainst",
    "giveawaysFor", "giveawaysAgainst",
    "faceOffsWonFor", "faceOffsWonAgainst",
]

# Keep only stats that actually exist
available_stats = [c for c in KEEP_STAT_COLS if c in mp_team.columns]

mp_team = mp_team[id_cols + available_stats].copy()

# Rename IDs
mp_team = mp_team.rename(columns={
    "gameId":        "game_id",
    "gameDate":      "game_date",
    "team":          "team_code",
    "playerTeam":    "team_code_dup",
    "opposingTeam":  "opp_code",
    "home_or_away":  "home_away",
})

mp_team["game_date"]  = pd.to_datetime(mp_team["game_date"])
mp_team["is_home"]    = (mp_team["home_away"].str.upper() == "HOME").astype(int)
mp_team["is_playoff"] = mp_team["playoffGame"].astype(int)

def has(col: str) -> bool:
    return col in mp_team.columns

show_mem()
print('step 2,3 done')
# =========================
# STEP 4: BASIC PER-GAME DERIVED METRICS
# =========================
if has("goalsFor") and has("goalsAgainst"):
    mp_team["goal_diff"] = mp_team["goalsFor"] - mp_team["goalsAgainst"]
    mp_team["win"] = (mp_team["goalsFor"] > mp_team["goalsAgainst"]).astype(int)
else:
    mp_team["goal_diff"] = np.nan
    mp_team["win"] = np.nan

for pct_col in ["xGoalsPercentage", "corsiPercentage", "fenwickPercentage"]:
    if has(pct_col):
        mp_team[pct_col] = mp_team[pct_col].astype(float)

show_mem()
print('step 4 done')
# =========================
# STEP 5: METRICS TO ROLL
# =========================
metrics_for_roll = [
    # scoreboard
    "goalsFor", "goalsAgainst", "goal_diff", "win",

    # xG core
    "xGoalsFor", "xGoalsAgainst",

    # shot volume / quality
    "shotsOnGoalFor", "shotsOnGoalAgainst",
    "shotAttemptsFor", "shotAttemptsAgainst",
    "highDangerShotsFor", "highDangerShotsAgainst",
    "totalShotCreditFor", "totalShotCreditAgainst",

    # advanced percentages
    "xGoalsPercentage", "corsiPercentage", "fenwickPercentage",
]

metrics_for_roll = [m for m in metrics_for_roll if has(m)]

show_mem()
print('step 5 done')
# =========================
# STEP 6: ROLLING FEATURES PER TEAM (optimized, no fragmentation)
# =========================
def add_team_rollings(df_team: pd.DataFrame) -> pd.DataFrame:
    df_team = df_team.sort_values("game_date").copy()
    df_team["games_played"] = np.arange(1, len(df_team) + 1)

    roll_frames = []

    # Rolling means + EWMs
    for col in metrics_for_roll:
        if col not in df_team.columns:
            continue

        sub = {}
        s = df_team[col]

        # Rolling windows
        for w in ROLL_WINDOWS:
            sub[f"{col}_roll{w}"] = s.rolling(window=w, min_periods=1).mean()

        # EWMs
        for label, alpha in EWM_ALPHAS.items():
            sub[f"{col}_ewm_{label}"] = s.ewm(alpha=alpha, adjust=False).mean()

        roll_frames.append(pd.DataFrame(sub, index=df_team.index))

    # Win% rollings
    if "win" in df_team.columns:
        win_sub = {}
        for w in [5, 10, 20]:
            win_sub[f"win_pct_roll{w}"] = df_team["win"].rolling(window=w, min_periods=1).mean()
        roll_frames.append(pd.DataFrame(win_sub, index=df_team.index))

    # Placeholder trend DF (we'll fill after concat)
    trend_sub = {}
    for base in ["xGoalsPercentage", "corsiPercentage", "fenwickPercentage"]:
        short = f"{base}_roll7"
        long  = f"{base}_roll30"
        # we'll compute trend later, once roll columns exist
        trend_sub[f"{base}_trend_7v30"] = np.nan
    trend_df = pd.DataFrame(trend_sub, index=df_team.index)
    roll_frames.append(trend_df)

    # Concat all roll features
    roll_all = pd.concat(roll_frames, axis=1)

    # Now compute trend features
    for base in ["xGoalsPercentage", "corsiPercentage", "fenwickPercentage"]:
        short = f"{base}_roll7"
        long  = f"{base}_roll30"
        if short in roll_all.columns and long in roll_all.columns:
            roll_all[f"{base}_trend_7v30"] = roll_all[short] - roll_all[long]

    # Combine original + all new features
    return pd.concat([df_team, roll_all], axis=1)

teams = []
for team_code, grp in mp_team.groupby("team_code"):
    teams.append(add_team_rollings(grp))

mp_team_roll = pd.concat(teams, ignore_index=True)

# Ensure unique (team, game) rows
mp_team_roll = mp_team_roll.sort_values(["team_code", "game_date", "game_id"])
mp_team_roll = mp_team_roll.drop_duplicates(
    subset=["team_code", "game_date", "game_id"],
    keep="first"
)

print("After dedupe team-level shape:", mp_team_roll.shape)


show_mem()
print('step 6 done')
# =========================
# STEP 7: STRENGTH-OF-SCHEDULE (SoS) FEATURES
# =========================

# 1) Define team strength metric (pre-game, lagged) using xG% roll30
if "xGoalsPercentage_roll30" in mp_team_roll.columns:
    mp_team_roll = mp_team_roll.sort_values(["team_code", "game_date"])
    mp_team_roll["team_strength_xg30"] = (
        mp_team_roll
        .groupby("team_code")["xGoalsPercentage_roll30"]
        .shift(1)  # pre-game strength (no leakage)
    )
else:
    mp_team_roll["team_strength_xg30"] = np.nan

# 2) Attach opponent strength per game
strength_cols = ["team_strength_xg30"]

strength_ref = (
    mp_team_roll[["game_id", "team_code"] + strength_cols]
    .rename(columns={"team_code": "opp_code"})
)

mp_team_roll = mp_team_roll.merge(
    strength_ref,
    on=["game_id", "opp_code"],
    how="left",
    suffixes=("", "_opp")
)

# 3) Rolling SoS features per team (how strong have my opponents been recently?)
sos_windows = [5, 10, 20]
sos_source_cols = ["team_strength_xg30_opp"]

mp_team_roll = mp_team_roll.sort_values(["team_code", "game_date"])

for col in sos_source_cols:
    if col not in mp_team_roll.columns:
        continue

    for w in sos_windows:
        sos_col = f"sos_xg30_roll{w}"
        mp_team_roll[sos_col] = (
            mp_team_roll
            .groupby("team_code")[col]
            .rolling(window=w, min_periods=1)
            .mean()
            .reset_index(level=0, drop=True)
        )

show_mem()
print('step 7 done')
# =========================
# STEP 7.5: DOWNSAMPLE NUMERIC TYPES TO SAVE MEMORY
# =========================
print("Before downcast memory:")
show_mem()

num_cols = mp_team_roll.select_dtypes(include=["float64", "int64"]).columns

for col in num_cols:
    col_max = mp_team_roll[col].max()
    col_min = mp_team_roll[col].min()
    if pd.api.types.is_float_dtype(mp_team_roll[col]):
        mp_team_roll[col] = mp_team_roll[col].astype("float32")
    else:
        # int downcast
        if col_min >= -32768 and col_max <= 32767:
            mp_team_roll[col] = mp_team_roll[col].astype("int16")
        elif col_min >= -2147483648 and col_max <= 2147483647:
            mp_team_roll[col] = mp_team_roll[col].astype("int32")
        else:
            # leave as int64 if it really needs it
            pass

print("After downcast memory:")
show_mem()
print('step 7.5 done')
# =========================
# STEP 8–10: CHUNKED GAME-LEVEL PIVOT + DIFF FEATURES + SAVE
# =========================

game_chunks = []

unique_seasons = sorted(mp_team_roll["season"].dropna().unique())
print("Seasons to process:", unique_seasons)

for season in unique_seasons:
    df_season = mp_team_roll[mp_team_roll["season"] == season].copy()
    print(f"\n=== Processing season {season} ===")
    print("Rows in this season team-level:", df_season.shape[0])
    show_mem()

    # Split into home / away
    home = df_season[df_season["is_home"] == 1].copy()
    away = df_season[df_season["is_home"] == 0].copy()

    merge_keys = ["game_id", "game_date", "season"]
    g = home.merge(
        away,
        on=merge_keys,
        suffixes=("_home", "_away")
    )

    # Canonical game date & season
    g["game_date"] = g["game_date"].dt.date
    g["season"]    = g["season"]

    # Team codes for later odds merge
    g["home_team_code"] = g["team_code_home"]
    g["away_team_code"] = g["team_code_away"]

    # Targets
    if {"goalsFor_home", "goalsFor_away"} <= set(g.columns):
        g["home_goals"] = g["goalsFor_home"]
        g["away_goals"] = g["goalsFor_away"]
        g["home_win"]   = (g["home_goals"] > g["away_goals"]).astype("int8")
        g["home_win_margin"] = g["home_goals"] - g["away_goals"]
        g["total_goals"] = g["home_goals"] + g["away_goals"]
    else:
        g["home_goals"] = np.nan
        g["away_goals"] = np.nan
        g["home_win"]   = np.nan
        g["home_win_margin"] = np.nan
        g["total_goals"] = np.nan

    # Playoff flag
    if "is_playoff_home" in g.columns and "is_playoff_away" in g.columns:
        g["is_playoff"] = g[["is_playoff_home", "is_playoff_away"]].max(axis=1)

    # ---- DIFFERENTIAL FEATURES (HOME - AWAY) FOR THIS SEASON ONLY ----
    all_cols = set(g.columns)
    home_metrics = [c for c in all_cols if c.endswith("_home")]
    away_metrics = [c for c in all_cols if c.endswith("_away")]
    base_names = set([c[:-5] for c in home_metrics]) & set([c[:-5] for c in away_metrics])

    diff_data = {}
    for base in base_names:
        h_col = f"{base}_home"
        a_col = f"{base}_away"
        diff_col = f"diff_{base}"

        if np.issubdtype(g[h_col].dtype, np.number) and np.issubdtype(g[a_col].dtype, np.number):
            diff_data[diff_col] = g[h_col] - g[a_col]

    if diff_data:
        diff_df = pd.DataFrame(diff_data, index=g.index)
        g = pd.concat([g, diff_df], axis=1)

    print(f"Season {season} game-level shape:", g.shape)
    show_mem()

    game_chunks.append(g)
    # Drop large intermediates to free memory
    del df_season, home, away, g, diff_df
    import gc; gc.collect()

# Concatenate all seasons
game_df = pd.concat(game_chunks, ignore_index=True)
del game_chunks
gc.collect()

print("\nFinal stats-only master shape:", game_df.shape)
show_mem()

# =========================
# SAVE STATS-ONLY MASTER
# =========================
OUTPUT_FILE.parent.mkdir(exist_ok=True, parents=True)
game_df.to_csv(OUTPUT_FILE, index=False)
print(f"Saved stats-only master dataset to: {OUTPUT_FILE}")



Team-level rows after filter: (44374, 111)
season
2008    1314
2009    1318
2010    1318
2011    1316
2012     806
Name: gameId, dtype: int64
Current memory usage: 5.26 GB
step 1 done
Current memory usage: 5.30 GB
step 2,3 done
Current memory usage: 5.30 GB
step 4 done
Current memory usage: 5.30 GB
step 5 done
After dedupe team-level shape: (44374, 168)
Current memory usage: 5.22 GB
step 6 done
Current memory usage: 5.28 GB
step 7 done
Before downcast memory:
Current memory usage: 5.28 GB
After downcast memory:
Current memory usage: 5.22 GB
step 7.5 done
Seasons to process: [2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]

=== Processing season 2008 ===
Rows in this season team-level: 2628
Current memory usage: 5.22 GB
Season 2008 game-level shape: (1314, 517)
Current memory usage: 5.22 GB

=== Processing season 2009 ===
Rows in this season team-level: 2636
Current memory usage: 5.22 GB
Season 2009 game-level shape: (1318, 517

In [16]:
import pandas as pd
import numpy as np
import gc

MASTER_FILE = "data/master.csv"
OUTPUT_TIER1 = "data/master_tier1.csv"

# =========================
# 1. Load master & sort
# =========================
master = pd.read_csv(MASTER_FILE, parse_dates=["game_date"])
print("Loaded master:", master.shape)

# Make sure we have the columns we expect
required_cols = {"game_id", "game_date", "season", "team_code_home", "team_code_away"}
missing = required_cols - set(master.columns)
if missing:
    raise ValueError(f"Master is missing expected columns: {missing}")

master = master.sort_values(["game_date", "game_id"]).reset_index(drop=True)

# =========================
# 2. Build long per-team game list
# =========================
base_cols = ["game_id", "game_date", "season"]

home_long = master[base_cols + ["team_code_home"]].rename(
    columns={"team_code_home": "team_code"}
)
home_long["is_home"] = 1

away_long = master[base_cols + ["team_code_away"]].rename(
    columns={"team_code_away": "team_code"}
)
away_long["is_home"] = 0

games_long = pd.concat([home_long, away_long], ignore_index=True)
games_long = games_long.sort_values(["team_code", "game_date", "game_id"]).reset_index(drop=True)

print("Long per-team games shape:", games_long.shape)

# =========================
# 3. Compute rest days and fatigue flags
# =========================
games_long["prev_game_date"] = games_long.groupby("team_code")["game_date"].shift(1)
games_long["days_rest"] = (games_long["game_date"] - games_long["prev_game_date"]).dt.days

# For first game of season / missing previous, treat as fully rested
games_long["days_rest"] = games_long["days_rest"].fillna(99)
games_long["days_rest"] = games_long["days_rest"].clip(lower=0)

# Back-to-back = 0 days rest (played yesterday)
games_long["is_b2b"] = (games_long["days_rest"] == 0).astype("int8")

# 3-in-4: this game + previous game(s) with <=1 day rest
prev_rest = games_long.groupby("team_code")["days_rest"].shift(1)
games_long["is_3in4"] = (
    (games_long["days_rest"] <= 1) &
    (prev_rest <= 1)
).fillna(False).astype("int8")

print("Sample fatigue rows:")
print(games_long.head())

# =========================
# 4. Split back to home/away and merge into master
# =========================
home_rest = (
    games_long[games_long["is_home"] == 1][
        ["game_id", "team_code", "days_rest", "is_b2b", "is_3in4"]
    ]
    .rename(
        columns={
            "team_code": "team_code_home",
            "days_rest": "home_days_rest",
            "is_b2b": "home_is_b2b",
            "is_3in4": "home_is_3in4",
        }
    )
)

away_rest = (
    games_long[games_long["is_home"] == 0][
        ["game_id", "team_code", "days_rest", "is_b2b", "is_3in4"]
    ]
    .rename(
        columns={
            "team_code": "team_code_away",
            "days_rest": "away_days_rest",
            "is_b2b": "away_is_b2b",
            "is_3in4": "away_is_3in4",
        }
    )
)

print("Home rest shape:", home_rest.shape)
print("Away rest shape:", away_rest.shape)

master = master.merge(home_rest, on=["game_id", "team_code_home"], how="left")
master = master.merge(away_rest, on=["game_id", "team_code_away"], how="left")

# =========================
# 5. Simple pace metric (game-level attempts)
# =========================
pace_cols = {"shotAttemptsFor_home", "shotAttemptsFor_away"}
if pace_cols.issubset(master.columns):
    master["game_pace_attempts"] = (
        master["shotAttemptsFor_home"].astype("float32") +
        master["shotAttemptsFor_away"].astype("float32")
    )
    print("Added game_pace_attempts.")
else:
    print("WARNING: pace columns not found; skipping pace metric.")

# =========================
# 6. Save Tier-1 enriched master
# =========================
print("Final master with Tier-1 features shape:", master.shape)
master.to_csv(OUTPUT_TIER1, index=False)
print(f"Saved Tier-1 enriched master to: {OUTPUT_TIER1}")

# Clean up
del games_long, home_rest, away_rest
gc.collect()


Loaded master: (22072, 517)
Long per-team games shape: (44144, 5)
Sample fatigue rows:
      game_id  game_date  season team_code  is_home prev_game_date  days_rest  \
0  2008020008 2008-10-09    2008       ANA        0            NaT       99.0   
1  2008020030 2008-10-12    2008       ANA        1     2008-10-09        3.0   
2  2008020042 2008-10-14    2008       ANA        0     2008-10-12        2.0   
3  2008020048 2008-10-15    2008       ANA        1     2008-10-14        1.0   
4  2008020061 2008-10-17    2008       ANA        1     2008-10-15        2.0   

   is_b2b  is_3in4  
0       0        0  
1       0        0  
2       0        0  
3       0        0  
4       0        0  
Home rest shape: (22072, 5)
Away rest shape: (22072, 5)
Added game_pace_attempts.
Final master with Tier-1 features shape: (22072, 524)
Saved Tier-1 enriched master to: data/master_tier1.csv


21645

In [24]:
# Update historical odds, grab odds for upcoming 14 days

In [34]:
import os
import math
import time
from datetime import datetime, timedelta, timezone

import numpy as np
import pandas as pd
import requests

# =========================
# CONFIG
# =========================

API_KEY = ("fd89ad7935c3f3851f9827d2bdaef90f")  # <-- set env var or hard-code
SPORT_KEY = "icehockey_nhl"

BOOKMAKERS = "draftkings,fanduel,pinnacle"
MARKETS = "h2h,spreads,totals"
ODDS_FORMAT = "american"
DATE_FORMAT = "iso"

MASTER_PATH = "data/master_tier1.csv"   # game-level stats
ODDS_PATH   = "data/odds.csv"           # historical odds store (existing + new)

# How far into the future to pull upcoming odds (for betting)
UPCOMING_DAYS = 14

# Historical snapshots: time of day (UTC) used as an approximate "near-closing" snapshot
SNAPSHOT_HOUR_UTC = 23   # 23:00Z on each game_date (tweak if you want earlier)


# =========================
# HELPER FUNCTIONS
# =========================

def get_json(url, params, sleep_on_429=True):
    """Small helper for GET requests with basic retry on 429."""
    while True:
        resp = requests.get(url, params=params, timeout=30)
        if resp.status_code == 200:
            return resp.json(), resp.headers
        if resp.status_code == 429 and sleep_on_429:
            # Hit rate limit, use headers to backoff if available
            reset = resp.headers.get("x-requests-reset")
            if reset:
                # reset is seconds until reset
                wait_s = int(reset) + 2
            else:
                wait_s = 60
            print(f"[429] Rate limited. Sleeping {wait_s}s...")
            time.sleep(wait_s)
            continue
        # Other error
        raise RuntimeError(
            f"Error from The Odds API: {resp.status_code} {resp.text}"
        )


def flatten_snapshot(snapshot_data, snapshot_ts_iso):
    """
    Flatten one historical snapshot's 'data' payload into rows
    compatible with our odds.csv schema:
      snapshot_ts, game_date, commence_time, event_id,
      home_team, away_team, book, market_type, team, odds, point
    """
    rows = []

    data = snapshot_data.get("data", snapshot_data)  # historical endpoint wraps in {timestamp, data} :contentReference[oaicite:3]{index=3}
    for ev in data:
        event_id = ev["id"]
        commence_time = ev["commence_time"]
        home_team = ev["home_team"]
        away_team = ev["away_team"]

        bookmakers = ev.get("bookmakers", [])
        for bm in bookmakers:
            book_key = bm["key"]
            markets = bm.get("markets", [])
            for mkt in markets:
                mkt_key = mkt["key"]  # "h2h", "spreads", "totals", etc.
                for outcome in mkt.get("outcomes", []):
                    team_name = outcome.get("name")
                    price = outcome.get("price")
                    point = outcome.get("point", np.nan)

                    rows.append(
                        {
                            "snapshot_ts": snapshot_ts_iso,
                            "game_date": pd.to_datetime(commence_time).date(),
                            "commence_time": commence_time,
                            "event_id": event_id,
                            "home_team": home_team,
                            "away_team": away_team,
                            "book": book_key,
                            "market_type": mkt_key,
                            "team": team_name,
                            "odds": price,
                            "point": float(point) if point is not None else np.nan,
                        }
                    )
    return rows


# =========================
# 1. LOAD MASTER + EXISTING ODDS
# =========================

# Load master game-level stats
master = pd.read_csv(MASTER_PATH, parse_dates=["game_date"])
print(f"Loaded master_tier1: {master.shape}")

# Load existing odds.csv (if it exists) with robust datetime parsing
if os.path.exists(ODDS_PATH):
    odds = pd.read_csv(ODDS_PATH)
    print(f"Loaded existing odds.csv (raw): {odds.shape}")

    # Make sure datetime-like columns are strings before parsing
    for col in ["game_date", "snapshot_ts", "commence_time"]:
        if col in odds.columns:
            odds[col] = odds[col].astype(str)

    # Robust parsing for ISO8601 + older styles (e.g. '2020-08-01 23:55:00+00:00')
    odds["game_date"] = pd.to_datetime(
        odds["game_date"],
        errors="coerce",
        utc=True,
        format="ISO8601",   # handles '2022-02-17T00:00:00Z' & friends
    ).dt.tz_convert(None)   # drop timezone, keep as naive datetime

    odds["snapshot_ts"] = pd.to_datetime(
        odds["snapshot_ts"],
        errors="coerce",
        utc=True,
        format="ISO8601",
    )

    odds["commence_time"] = pd.to_datetime(
        odds["commence_time"],
        errors="coerce",
        utc=True,
        format="ISO8601",
    )

    print(f"Loaded existing odds.csv (parsed datetimes): {odds.shape}")
else:
    odds = pd.DataFrame(
        columns=[
            "snapshot_ts",
            "game_date",
            "commence_time",
            "event_id",
            "home_team",
            "away_team",
            "book",
            "market_type",
            "team",
            "odds",
            "point",
        ]
    )
    print("No existing odds.csv found. Starting fresh.")

# Determine date ranges
master_max_date = master["game_date"].max().normalize()
today_utc = datetime.now(timezone.utc).date()

print(f"Max game_date in master: {master_max_date}")
print(f"Today (UTC date):        {today_utc}")

if len(odds) > 0 and "game_date" in odds.columns:
    last_odds_date = odds["game_date"].max().normalize()
else:
    # If no odds yet, start from earliest master game_date
    last_odds_date = master["game_date"].min().normalize()

print(f"Last game_date in odds   : {last_odds_date}")

# Historical backfill window (only for completed games up to 'today')

# Normalize both to pure dates (datetime.date)
hist_start_date = (last_odds_date + timedelta(days=1)).date()
hist_end_date = min(master_max_date.date(), today_utc)

print(f"Normalized hist_start_date: {hist_start_date} ({type(hist_start_date)})")
print(f"Normalized hist_end_date:   {hist_end_date} ({type(hist_end_date)})")

if hist_start_date > hist_end_date:
    print("No new historical dates to backfill (odds already up to date).")
    hist_dates = []
else:
    hist_dates = pd.date_range(hist_start_date, hist_end_date, freq="D").date
    print(
        f"Historical backfill dates: {hist_start_date} → {hist_end_date} "
        f"({len(hist_dates)} days)"
    )



# =========================
# 2. HISTORICAL BACKFILL LOOP
# =========================

new_hist_rows = []

for d in hist_dates:
    # snapshot at SNAPSHOT_HOUR_UTC on that calendar date (approx near-closing)
    snapshot_dt = datetime(d.year, d.month, d.day, SNAPSHOT_HOUR_UTC, 0, 0, tzinfo=timezone.utc)
    snapshot_iso = snapshot_dt.strftime("%Y-%m-%dT%H:%M:%SZ")

    params = {
        "apiKey": API_KEY,
        "sports": SPORT_KEY,  # some docs use 'sport', some 'sports'; we'll use 'sport' below
    }

    # Historical endpoint for featured markets:
    # /v4/historical/sports/{sport}/odds?date=... :contentReference[oaicite:4]{index=4}
    url = f"https://api.the-odds-api.com/v4/historical/sports/{SPORT_KEY}/odds"
    params = {
        "apiKey": API_KEY,
        "date": snapshot_iso,
        "markets": MARKETS,
        "bookmakers": BOOKMAKERS,
        "oddsFormat": ODDS_FORMAT,
        "dateFormat": DATE_FORMAT,
    }

    print(f"\nFetching historical snapshot for {d} at {snapshot_iso} ...")
    try:
        snapshot_json, headers = get_json(url, params)
    except RuntimeError as e:
        print(f"  Error on {d}: {e}")
        continue

    # snapshot_json is {timestamp, previous_timestamp, next_timestamp, data: [...]}
    ts = snapshot_json.get("timestamp", snapshot_iso)
    rows = flatten_snapshot(snapshot_json, ts)
    print(f"  Rows from snapshot: {len(rows)}")

    # Optional: filter to only games that exist in master and have game_date == d
    if rows:
        df_rows = pd.DataFrame(rows)
        # Join to master on (game_date, home, away) if you want to ensure only known games
        # For now, keep all; you can further filter here if desired.
        new_hist_rows.extend(df_rows.to_dict("records"))

        # show usage headers if available
        used = headers.get("x-requests-used")
        remaining = headers.get("x-requests-remaining")
        if used and remaining:
            print(f"  Requests used: {used}, remaining: {remaining}")

# Convert new historical rows
if new_hist_rows:
    new_hist_df = pd.DataFrame(new_hist_rows)
    new_hist_df["game_date"] = pd.to_datetime(new_hist_df["game_date"])
    new_hist_df["snapshot_ts"] = pd.to_datetime(new_hist_df["snapshot_ts"])
    new_hist_df["commence_time"] = pd.to_datetime(new_hist_df["commence_time"])
    print(f"\nNew historical odds rows: {new_hist_df.shape}")
else:
    new_hist_df = pd.DataFrame(columns=odds.columns)
    print("\nNo new historical odds rows fetched.")


# =========================
# 3. UPCOMING ODDS (NEXT 2 WEEKS)
# =========================

upcoming_rows = []

upcoming_until = datetime.now(timezone.utc) + timedelta(days=UPCOMING_DAYS)
upcoming_until_iso = upcoming_until.strftime("%Y-%m-%dT%H:%M:%SZ")

url_upcoming = f"https://api.the-odds-api.com/v4/sports/{SPORT_KEY}/odds"
params_upcoming = {
    "apiKey": API_KEY,
    "bookmakers": BOOKMAKERS,
    "markets": MARKETS,
    "oddsFormat": ODDS_FORMAT,
    "dateFormat": DATE_FORMAT,
    "commenceTimeTo": upcoming_until_iso,  # only games starting in next N days :contentReference[oaicite:5]{index=5}
}

print(f"\nFetching upcoming odds for next {UPCOMING_DAYS} days (until {upcoming_until_iso})...")
try:
    up_json, up_headers = get_json(url_upcoming, params_upcoming)
    # up_json is a plain list of events for the regular /odds endpoint :contentReference[oaicite:6]{index=6}
    rows = flatten_snapshot({"data": up_json}, datetime.now(timezone.utc).isoformat())
    print(f"Upcoming events rows: {len(rows)}")
    upcoming_rows.extend(rows)
except RuntimeError as e:
    print(f"  Error fetching upcoming odds: {e}")

if upcoming_rows:
    upcoming_df = pd.DataFrame(upcoming_rows)
    upcoming_df["game_date"] = pd.to_datetime(upcoming_df["game_date"])
    upcoming_df["snapshot_ts"] = pd.to_datetime(upcoming_df["snapshot_ts"])
    upcoming_df["commence_time"] = pd.to_datetime(upcoming_df["commence_time"])
    print(f"Upcoming odds df shape: {upcoming_df.shape}")
else:
    upcoming_df = pd.DataFrame(columns=odds.columns)
    print("No upcoming odds rows fetched.")


# =========================
# 4. COMBINE WITH EXISTING ODDS + DEDUPE
# =========================

all_new = pd.concat([new_hist_df, upcoming_df], ignore_index=True)
print(f"\nTotal new odds rows (historical + upcoming): {all_new.shape}")

if not all_new.empty:
    # Ensure same columns as existing
    if odds.empty:
        combined = all_new.copy()
    else:
        # Union of columns
        all_cols = sorted(set(odds.columns).union(all_new.columns))
        odds = odds.reindex(columns=all_cols)
        all_new = all_new.reindex(columns=all_cols)
        combined = pd.concat([odds, all_new], ignore_index=True)

    # Dedupe:
    # sort by snapshot_ts so that the latest snapshot is kept
    combined = combined.sort_values("snapshot_ts")

    dedupe_cols = [
        "game_date",
        "commence_time",
        "event_id",
        "home_team",
        "away_team",
        "book",
        "market_type",
        "team",
        "point",
    ]

    before = combined.shape[0]
    combined = combined.drop_duplicates(subset=dedupe_cols, keep="last")
    after = combined.shape[0]

    print(f"Combined odds rows before dedupe: {before}")
    print(f"Combined odds rows after  dedupe: {after}")

    # Sort by game_date and commence_time for readability
    combined = combined.sort_values(["game_date", "commence_time", "book", "market_type"]).reset_index(drop=True)

    # Save back to odds.csv
    combined.to_csv(ODDS_PATH, index=False)
    print(f"\n✅ Saved updated odds to: {ODDS_PATH}")
else:
    print("\nNo new odds to add; odds.csv unchanged.")


Loaded master_tier1: (22072, 524)
Loaded existing odds.csv (raw): (149790, 11)
Loaded existing odds.csv (parsed datetimes): (149790, 11)
Max game_date in master: 2025-11-24 00:00:00
Today (UTC date):        2025-11-26
Last game_date in odds   : 2024-06-24 00:00:00
Normalized hist_start_date: 2024-06-25 (<class 'datetime.date'>)
Normalized hist_end_date:   2025-11-24 (<class 'datetime.date'>)
Historical backfill dates: 2024-06-25 → 2025-11-24 (518 days)

Fetching historical snapshot for 2024-06-25 at 2024-06-25T23:00:00Z ...
  Rows from snapshot: 12
  Requests used: 41010, remaining: 58990

Fetching historical snapshot for 2024-06-26 at 2024-06-26T23:00:00Z ...
  Rows from snapshot: 12
  Requests used: 41040, remaining: 58960

Fetching historical snapshot for 2024-06-27 at 2024-06-27T23:00:00Z ...
  Rows from snapshot: 12
  Requests used: 41070, remaining: 58930

Fetching historical snapshot for 2024-06-28 at 2024-06-28T23:00:00Z ...
  Rows from snapshot: 12
  Requests used: 41100, rema

In [48]:
import pandas as pd
import numpy as np

# =========================
# PATHS
# =========================
MASTER_PATH = "data/master_tier1.csv"
ODDS_PATH = "data/odds.csv"
OUTPUT_PATH = "data/master_with_odds.csv"

# =========================
# TEAM NORMALIZATION
# =========================

def normalize_team_code(code: str) -> str:
    """
    Example inputs:
      'S.J' -> 'SJ'
      'T.B' -> 'TB'
      'ANA' -> 'ANA'
    """
    if pd.isna(code):
        return None
    return str(code).replace(".", "").replace(" ", "").upper()

TEAM_NAME_MAP = {
    "ANA": "Anaheim Ducks",
    "ARI": "Arizona Coyotes",
    "BOS": "Boston Bruins",
    "BUF": "Buffalo Sabres",
    "CAR": "Carolina Hurricanes",
    "CBJ": "Columbus Blue Jackets",
    "CGY": "Calgary Flames",
    "CHI": "Chicago Blackhawks",
    "COL": "Colorado Avalanche",
    "DAL": "Dallas Stars",
    "DET": "Detroit Red Wings",
    "EDM": "Edmonton Oilers",
    "FLA": "Florida Panthers",
    "LAK": "Los Angeles Kings",
    "MIN": "Minnesota Wild",
    "MTL": "Montreal Canadiens",
    "NJD": "New Jersey Devils",
    "NSH": "Nashville Predators",
    "NYI": "New York Islanders",
    "NYR": "New York Rangers",
    "OTT": "Ottawa Senators",
    "PHI": "Philadelphia Flyers",
    "PIT": "Pittsburgh Penguins",
    "SEA": "Seattle Kraken",
    "SJS": "San Jose Sharks",
    "STL": "St. Louis Blues",
    "TBL": "Tampa Bay Lightning",
    "TB":  "Tampa Bay Lightning",  # sometimes TB
    "TOR": "Toronto Maple Leafs",
    "VAN": "Vancouver Canucks",
    "VGK": "Vegas Golden Knights",
    "WPG": "Winnipeg Jets",
    "WSH": "Washington Capitals",
}

def convert_team_code_to_full_name(code_series: pd.Series) -> pd.Series:
    """
    MoneyPuck-style code → full team name.
    If value isn't a code (e.g. already 'Calgary Flames'), returns original.
    """
    def _map_one(x):
        if pd.isna(x):
            return x
        norm = normalize_team_code(x)
        return TEAM_NAME_MAP.get(norm, x)
    return code_series.apply(_map_one)

# =========================
# HELPER: AMERICAN ODDS → IMPLIED PROB
# =========================

def american_to_prob(x):
    if pd.isna(x):
        return np.nan
    x = float(x)
    if x < 0:
        return -x / (-x + 100.0)
    else:
        return 100.0 / (x + 100.0)

# =========================
# 1. LOAD MASTER + ENSURE game_date
# =========================

master = pd.read_csv(MASTER_PATH)
print("Loaded master_tier1:", master.shape)
print("Master columns sample:", list(master.columns)[:30])

# Handle game_date / gameDate / other variants
if "game_date" in master.columns:
    master["game_date"] = pd.to_datetime(master["game_date"])
elif "gameDate" in master.columns:
    master["game_date"] = pd.to_datetime(master["gameDate"])
else:
    raise ValueError(
        "Could not find 'game_date' or 'gameDate' in master_tier1. "
        f"Available columns: {list(master.columns)}"
    )

# Ensure we have home/away team codes
if not {"team_code_home", "opp_code_home"}.issubset(master.columns):
    raise ValueError(
        "Expected columns 'team_code_home' and 'opp_code_home' in master_tier1 "
        f"but got: {list(master.columns)}"
    )

master["home_team"] = convert_team_code_to_full_name(master["team_code_home"])
master["away_team"] = convert_team_code_to_full_name(master["opp_code_home"])

print("Sample home/away from master:")
print(master[["game_date", "team_code_home", "opp_code_home", "home_team", "away_team"]].head())

# =========================
# 2. LOAD ODDS + BASIC CLEANUP
# =========================

odds = pd.read_csv(ODDS_PATH)
print("Raw odds shape:", odds.shape)
print("Odds dtypes before:", odds.dtypes)

# Parse game_date
odds["game_date"] = pd.to_datetime(odds["game_date"], errors="coerce").dt.normalize()

# Standardize book names
odds["book"] = odds["book"].astype(str).str.lower()
books = ["pinnacle", "fanduel", "draftkings"]
odds = odds[odds["book"].isin(books)]
print("Filtered odds to target books:", odds.shape)

# Ensure team columns are strings
for col in ["home_team", "away_team", "team"]:
    if col in odds.columns:
        odds[col] = odds[col].astype(str)

# Pass through same converter; if already full names, it just leaves them as-is
odds["home_team"] = convert_team_code_to_full_name(odds["home_team"])
odds["away_team"] = convert_team_code_to_full_name(odds["away_team"])
odds["team"] = convert_team_code_to_full_name(odds["team"])

print("Sample odds rows:")
print(odds.head())

# =========================
# 3. PIVOT MONEYLINE ODDS
# =========================

ml = odds[odds["market_type"] == "h2h"].copy()

if ml.empty:
    print("No H2H (ML) rows in odds.csv!")
    ml_pivot = pd.DataFrame(columns=["game_date", "home_team", "away_team"])
else:
    ml["home_flag"] = (ml["team"] == ml["home_team"]).astype(int)
    ml["side"] = ml["home_flag"].map({1: "home", 0: "away"})

    ml_pivot = ml.pivot_table(
        index=["game_date", "home_team", "away_team"],
        columns=["book", "side"],
        values="odds",
        aggfunc="last",
    )

    ml_pivot.columns = [
        f"{book}_ml_{side}" for (book, side) in ml_pivot.columns
    ]
    ml_pivot = ml_pivot.reset_index()

print("ML pivot shape:", ml_pivot.shape)
print("ML pivot cols:", [c for c in ml_pivot.columns if "ml_" in c][:10])

# =========================
# 4. PIVOT TOTALS ODDS
# =========================

tot = odds[odds["market_type"] == "totals"].copy()

if not tot.empty:
    tot["side"] = tot["team"].str.lower()

    tot_pivot = tot.pivot_table(
        index=["game_date", "home_team", "away_team", "point"],
        columns=["book", "side"],
        values="odds",
        aggfunc="last",
    ).reset_index()

    tot_pivot.columns = [
        str(c).replace(" ", "_").lower() for c in tot_pivot.columns
    ]
    print("Totals pivot shape:", tot_pivot.shape)
    print("Totals pivot columns:", list(tot_pivot.columns))
else:
    tot_pivot = None
    print("No totals rows in odds.csv")

# =========================
# 5. MERGE ODDS INTO MASTER
# =========================

df = master.copy()

# Merge ML odds
df = df.merge(
    ml_pivot,
    on=["game_date", "home_team", "away_team"],
    how="left",
)

# Merge totals odds, ONLY if the necessary keys exist
if (
    tot_pivot is not None
    and all(k in tot_pivot.columns for k in ["game_date", "home_team", "away_team"])
):
    df = df.merge(
        tot_pivot,
        on=["game_date", "home_team", "away_team"],
        how="left",
        suffixes=("", "_totals"),
    )
else:
    if tot_pivot is None:
        print("Skipping totals merge: no totals pivot.")
    else:
        print(
            "Skipping totals merge: expected merge keys missing in tot_pivot. "
            f"Columns present: {list(tot_pivot.columns)}"
        )

print("After merging odds, shape:", df.shape)

# =========================
# 6. ADD IMPLIED PROBABILITIES FOR ML
# =========================

for book in books:
    for side in ["home", "away"]:
        col = f"{book}_ml_{side}"
        if col in df.columns:
            df[f"imp_prob_{book}_{side}"] = df[col].apply(american_to_prob)

print("Added implied probability columns for ML.")

# =========================
# 7. SAVE FINAL DATASET
# =========================

df.to_csv(OUTPUT_PATH, index=False)
print(f"✅ Saved master with odds to: {OUTPUT_PATH}")

# Quick sanity checks
ml_cols = [c for c in df.columns if c.endswith("_ml_home")]
print("Example ML cols:", ml_cols[:10])

games_with_any_ml = df[ml_cols].notna().any(axis=1).sum() if ml_cols else 0
print("Games with any ML odds:", games_with_any_ml)

print("Max game_date overall:", df["game_date"].max())
if "pinnacle_ml_home" in df.columns:
    print("Max game_date with Pinnacle ML:", df[~df["pinnacle_ml_home"].isna()]["game_date"].max())


Loaded master_tier1: (22072, 524)
Master columns sample: ['game_id', 'game_date', 'season', 'team_code_home', 'team_code_dup_home', 'opp_code_home', 'home_away_home', 'playoffGame_home', 'goalsFor_home', 'goalsAgainst_home', 'shotsOnGoalFor_home', 'shotsOnGoalAgainst_home', 'shotAttemptsFor_home', 'shotAttemptsAgainst_home', 'blockedShotAttemptsFor_home', 'blockedShotAttemptsAgainst_home', 'missedShotsFor_home', 'missedShotsAgainst_home', 'highDangerShotsFor_home', 'highDangerShotsAgainst_home', 'mediumDangerShotsFor_home', 'mediumDangerShotsAgainst_home', 'lowDangerShotsFor_home', 'lowDangerShotsAgainst_home', 'totalShotCreditFor_home', 'totalShotCreditAgainst_home', 'scoreAdjustedTotalShotCreditFor_home', 'scoreAdjustedTotalShotCreditAgainst_home', 'xGoalsFor_home', 'xGoalsAgainst_home']
Sample home/away from master:
   game_date team_code_home opp_code_home            home_team  \
0 2008-10-04            T.B           NYR  Tampa Bay Lightning   
1 2008-10-04            OTT          

In [54]:
import pandas as pd

INPUT_PATH = "data/master_with_odds.csv"
OUTPUT_PATH = "data/master_cleaned.csv"

# =========================
# 1. LOAD DATA
# =========================
df = pd.read_csv(INPUT_PATH)
print("Loaded:", INPUT_PATH, "shape:", df.shape)

# =========================
# 2. CONSTANT / ALL-NaN COLUMNS
# =========================
# Columns with <=1 unique value (including NaN)
nunique = df.nunique(dropna=False)
constant_cols = nunique[nunique <= 1].index.tolist()

print(f"\nFound {len(constant_cols)} constant / all-NaN columns:")
print(constant_cols[:40], "..." if len(constant_cols) > 40 else "")

# =========================
# 3. DUPLICATE COLUMNS (SAME VALUES)
# =========================
# This treats columns as rows (via transpose) and finds duplicates
# Safe for ~600 cols x 22k rows
dup_mask = df.T.duplicated(keep="first")
duplicate_cols = df.columns[dup_mask].tolist()

print(f"\nFound {len(duplicate_cols)} duplicate columns (same values as another column):")
print(duplicate_cols[:40], "..." if len(duplicate_cols) > 40 else "")

# =========================
# 4. MANUALLY-REDUNDANT COLUMNS
# =========================
# These are things we know are useless or clearly duplicated in meaning.
manual_candidates = [
    "home_away_home",
    "home_away_away",
    # sometimes we had helper/dup cols in earlier processing:
    "team_code_dup_home",
    "team_code_dup_away",
    "gameDate",          # older raw date column if still present
    # if season appears multiple ways, we only need the primary:
    "season_home",
    "season_away",
]

manual_to_drop = [c for c in manual_candidates if c in df.columns]

print(f"\nManual obvious junk columns present in df: {len(manual_to_drop)}")
print(manual_to_drop)

# =========================
# 5. BUILD FINAL DROP LIST
# =========================
garbage_cols = sorted(set(constant_cols) | set(duplicate_cols) | set(manual_to_drop))

print(f"\nTotal columns to drop: {len(garbage_cols)}")
print(garbage_cols[:60], "..." if len(garbage_cols) > 60 else "")

# Safety: don’t nuke the df by accident
if len(garbage_cols) > 0:
    df_clean = df.drop(columns=garbage_cols)
else:
    df_clean = df.copy()

print("\nOriginal shape:", df.shape)
print("Cleaned  shape:", df_clean.shape)
print("Columns removed:", len(df.columns) - len(df_clean.columns))

# =========================
# 6. SAVE CLEANED VERSION
# =========================
df_clean.to_csv(OUTPUT_PATH, index=False)
print(f"\n✅ Saved cleaned master to: {OUTPUT_PATH}")


Loaded: data/master_with_odds.csv shape: (22072, 538)

Found 10 constant / all-NaN columns:
['home_away_home', 'is_home_home', 'home_away_away', 'is_home_away', 'diff_is_playoff', 'diff_playoffGame', 'diff_is_home', 'home_is_b2b', 'home_is_3in4', 'away_is_b2b'] 

Found 68 duplicate columns (same values as another column):
['team_code_dup_home', 'is_playoff_home', 'team_code_away', 'team_code_dup_away', 'opp_code_away', 'playoffGame_away', 'goalsFor_away', 'goalsAgainst_away', 'shotsOnGoalFor_away', 'shotsOnGoalAgainst_away', 'shotAttemptsFor_away', 'shotAttemptsAgainst_away', 'blockedShotAttemptsFor_away', 'blockedShotAttemptsAgainst_away', 'missedShotsFor_away', 'missedShotsAgainst_away', 'highDangerShotsFor_away', 'highDangerShotsAgainst_away', 'mediumDangerShotsFor_away', 'mediumDangerShotsAgainst_away', 'lowDangerShotsFor_away', 'lowDangerShotsAgainst_away', 'totalShotCreditFor_away', 'totalShotCreditAgainst_away', 'scoreAdjustedTotalShotCreditFor_away', 'scoreAdjustedTotalShotCred